In [0]:
from pyspark.sql.functions import when, col, count
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt
import pandas as pd

## Load data

In [0]:
clean_data = spark.table('workspace.telco.bronze_data')

In [0]:
display(clean_data.limit(5))

## EDA

### Number of records

In [0]:
# Number of records
clean_data.select(count("customerID")).show()

### Distribution of the Categorical Columns

In [0]:
def plot_cat_distribution(df:pd.DataFrame, column:str):
    """ 
        Plot the distribution of a column in a dataframe
        Input:
            df (pandas dataframe): Data
            column (str): column name
        
    """
    # Column count
    df.groupBy(column).count().show()
    # Churn Distribution - Pie chart
    column_count = df.groupBy(column).count().toPandas()
    plt.figure(figsize=(10, 5))
    plt.pie(data=column_count, x="count", labels=column, autopct="%.2f%%");
    plt.show()

In [0]:
categorical_cols_1 = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity"]
categorical_cols_2 = ["OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "Churn"]
categorical_cols = categorical_cols_1 + categorical_cols_2
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

In [0]:
for cat in categorical_cols:
    print(20*"*" + f" Column: {cat} " + 20*"*")
    plot_cat_distribution(clean_data, cat)

### Numerical Distribution

In [0]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="MonthlyCharges", hue="Contract")
plt.title("Monthly Charges by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Monthly Charges")
plt.show()

In [0]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="TotalCharges", hue="Contract")
plt.title("Total Charges by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Total Charges")
plt.show()

In [0]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="tenure", hue="Contract")
plt.title("Tenure by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Tenure")
plt.show()

## Correlation among features

### Encoding categorical features

In [0]:
display(clean_data.limit(5))

In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

clean_data_df = clean_data.toPandas()

for col in categorical_cols:
    clean_data_df[col] = le.fit_transform(clean_data_df[col])
display(clean_data_df.head())


In [0]:
del clean_data
clean_data = spark.createDataFrame(clean_data_df)
clean_data.write.option("overwrite", True).saveAsTable("workspace.telco.indexed_data")
display(clean_data.limit(5)) 

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Assemble the features into a single vector column
assembler = VectorAssembler(
    inputCols=["tenure", "MonthlyCharges", "TotalCharges"],
    outputCol="features"
)
vector_data = assembler.transform(clean_data).select("features")

# Compute the Pearson correlation matrix
corr_matrix = Correlation.corr(vector_data, "features", "pearson").head()

# The result is a DenseMatrix
print("Pearson correlation matrix:\n" + str(corr_matrix[0]))


In [0]:
import numpy as np
import pandas as pd

# Extract the DenseMatrix from the Row
corr_array = np.array(corr_matrix[0].toArray())

# Convert to Pandas DataFrame
corr_df = pd.DataFrame(corr_array, columns=numerical_cols, index=numerical_cols)

# Display nicely in Databricks
display(corr_df)


In [0]:
plt.figure(figsize=(10,10))
sns.heatmap(corr_df, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [0]:
import math
from pyspark.sql.functions import col
import scipy.stats as ss

def cramers_v(df, col1, col2):
    contingency = df.groupBy(col1, col2).count().toPandas().pivot(index=col1, columns=col2, values='count').fillna(0)
    chi2 = ss.chi2_contingency(contingency)[0]       # chi-square
    n = contingency.sum().sum()
    phi2 = chi2 / n
    r, k = contingency.shape
    return math.sqrt(phi2 / min(k - 1, r - 1))       # Cramér's V


In [0]:
import pandas as pd
import scipy.stats as ss

results = {}

for c1 in categorical_cols:
    results[c1] = {}
    for c2 in categorical_cols:
        if c1 == c2:
            results[c1][c2] = 1.0
        else:
            results[c1][c2] = cramers_v(clean_data, c1, c2)

cat_corr_df = pd.DataFrame(results)
display(cat_corr_df)   # Databricks table view

In [0]:
plt.figure(figsize=(12,10))
sns.heatmap(cat_corr_df, annot=True, cmap="Blues", fmt=".2f", 
            annot_kws={"size":8})  # smaller annotation font
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title("Cramér's V — Categorical Correlation")
plt.show()

In [0]:
from scipy.stats import pointbiserialr

def point_biserial(df, cat, num):
    pdf = df.select(cat, num).dropna().toPandas()
    return pointbiserialr(pdf[cat], pdf[num]).correlation


In [0]:
import numpy as np

def eta_squared(df, cat, num):
    pdf = df.select(cat, num).dropna().toPandas()
    categories = pdf[cat].unique()
    
    y = pdf[num].values
    grand_mean = np.mean(y)

    ss_between = sum([
        len(pdf[pdf[cat] == c]) * (np.mean(pdf[pdf[cat] == c][num]) - grand_mean)**2
        for c in categories
    ])
    ss_total = sum((y - grand_mean)**2)

    return ss_between / ss_total if ss_total > 0 else 0


In [0]:
import pandas as pd

results = {}

for cat in categorical_cols:
    results[cat] = {}
    unique_count = clean_data.select(cat).distinct().count()

    for num in numerical_cols:
        if unique_count == 2:   # binary → point biserial
            results[cat][num] = point_biserial(clean_data, cat, num)
        else:                   # multiclass → correlation ratio
            results[cat][num] = eta_squared(clean_data, cat, num)

mixed_corr_df = pd.DataFrame(results).T
display(mixed_corr_df)


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.heatmap(mixed_corr_df, annot=True, cmap="coolwarm")
plt.title("Mixed Correlation: Numerical × Categorical Features")
plt.show()
